# 07 - Simple Quantitative Smoke Tests

This notebook runs a small executable smoke test for the attribution methods that will be presented in the report:

- `level3`
- `legrad_final_score_relu_attention_gradient_mean_layers_source_tokens`
- `legrad_final_score_relu_attention_gradient_renormalized_hubert_layer_weighted_source_tokens`
- `legrad_layer_local_score_relu_attention_gradient_mean_layers_source_tokens`
- `legrad_layer_local_score_relu_attention_gradient_renormalized_hubert_layer_weighted_source_tokens`

The goal is not to reproduce the full quantitative evaluation. The goal is to verify that the report methods run end-to-end in the faithfulness and class-specificity scripts on local audio.

In [1]:
from pathlib import Path
import subprocess
import sys
import zipfile

import pandas as pd

ROOT = Path.cwd().resolve()
while not (ROOT / "src").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

OUTPUT_ROOT = ROOT / "outputs" / "test_07_report_methods_smoke"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

REPORT_METHODS = [
    "level3",
    "legrad_final_score_relu_attention_gradient_mean_layers_source_tokens",
    "legrad_final_score_relu_attention_gradient_renormalized_hubert_layer_weighted_source_tokens",
    "legrad_layer_local_score_relu_attention_gradient_mean_layers_source_tokens",
    "legrad_layer_local_score_relu_attention_gradient_renormalized_hubert_layer_weighted_source_tokens",
]
METHODS_ARG = ",".join(REPORT_METHODS)

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 180)
print("Project root:", ROOT)
print("Output root:", OUTPUT_ROOT)
print("Methods:")
for method in REPORT_METHODS:
    print("-", method)


Project root: C:\Users\mateu\repos\gradient_based_speach_xai
Output root: C:\Users\mateu\repos\gradient_based_speach_xai\outputs\test_07_report_methods_smoke
Methods:
- level3
- legrad_final_score_relu_attention_gradient_mean_layers_source_tokens
- legrad_final_score_relu_attention_gradient_renormalized_hubert_layer_weighted_source_tokens
- legrad_layer_local_score_relu_attention_gradient_mean_layers_source_tokens
- legrad_layer_local_score_relu_attention_gradient_renormalized_hubert_layer_weighted_source_tokens


## Build A Small Prediction Manifest

The evaluation scripts require a `predictions.csv` with correctly classified examples. This cell builds a small local manifest from the already generated duration-matched result ZIPs and rewrites `/content/...` paths to the local `data/` directory.

In [2]:
def local_audio_path(row: pd.Series) -> str | None:
    relative_path = str(row["relative_path"]).replace("\\", "/")
    dataset = str(row["dataset"])
    if dataset == "IEMOCAP":
        candidates = [ROOT / "data" / relative_path]
    elif dataset == "RAVDESS":
        candidates = [
            ROOT / "data" / "ravdess" / relative_path,
            ROOT / "data" / "ravdess" / "Audio_Speech_Actors_01-24" / relative_path,
        ]
    else:
        candidates = [ROOT / relative_path]
    for candidate in candidates:
        if candidate.exists():
            return str(candidate.resolve())
    return None


def build_smoke_predictions() -> Path:
    sources = [
        (
            ROOT / "IEMOCAP_speech_xai_results_20260711_074459.zip",
            "iemocap_ravdess_duration_matched_speechxai_20260710_194505_901788/duration_matched_records.csv",
        ),
        (
            ROOT / "RAVDESS_speech_xai_results_20260711_100706.zip",
            "ravdess_duration_matched_speechxai_20260711_092556_063830/duration_matched_records.csv",
        ),
    ]
    usecols = [
        "dataset", "audio_path", "audio_id", "relative_path", "true_class", "true_label",
        "session_id", "dialogue_id", "utterance_id", "iemocap_emotion", "annotation_path",
        "actor_id", "ravdess_emotion", "original_class", "original_label", "original_confidence",
    ]
    frames = []
    for zip_path, inner_path in sources:
        if not zip_path.exists():
            raise FileNotFoundError(zip_path)
        with zipfile.ZipFile(zip_path) as archive:
            with archive.open(inner_path) as handle:
                header = pd.read_csv(handle, nrows=0).columns.tolist()
                columns = [column for column in usecols if column in header]
            with archive.open(inner_path) as handle:
                frame = pd.read_csv(handle, usecols=columns, low_memory=False)
        frame = frame[frame["original_class"].eq(frame["true_class"])].copy()
        frame = frame.drop_duplicates(["dataset", "audio_id"])
        frames.append(frame)

    records = pd.concat(frames, ignore_index=True)
    records["audio_path"] = records.apply(local_audio_path, axis=1)
    records = records[records["audio_path"].notna()].copy()
    records["predicted_class"] = records["original_class"].astype(int)
    records["predicted_label"] = records["original_label"].astype(str)
    records["confidence"] = records["original_confidence"].astype(float)
    records["is_correct"] = records["predicted_class"].eq(records["true_class"].astype(int))

    selected_parts = []
    for dataset in ["IEMOCAP", "RAVDESS"]:
        subset = records[records["dataset"].eq(dataset)].copy()
        for class_id in sorted(subset["true_class"].unique()):
            selected_parts.append(subset[subset["true_class"].eq(class_id)].sort_values("audio_id").head(1))
    selected = pd.concat(selected_parts, ignore_index=True).head(8)

    base_columns = [
        "dataset", "audio_path", "relative_path", "audio_id", "true_class", "true_label",
        "predicted_class", "predicted_label", "is_correct", "confidence", "session_id",
        "dialogue_id", "utterance_id", "iemocap_emotion", "annotation_path", "actor_id",
        "ravdess_emotion",
    ]
    selected = selected[[column for column in base_columns if column in selected.columns]]
    output_path = OUTPUT_ROOT / "report_methods_smoke_predictions.csv"
    selected.to_csv(output_path, index=False)
    return output_path

predictions_csv = build_smoke_predictions()
predictions = pd.read_csv(predictions_csv)
print(predictions_csv)
display(predictions[["dataset", "audio_id", "true_label", "predicted_label", "audio_path"]])
print("Rows:", len(predictions), "Correct:", int(predictions["is_correct"].sum()))


C:\Users\mateu\repos\gradient_based_speach_xai\outputs\test_07_report_methods_smoke\report_methods_smoke_predictions.csv


,dataset,audio_id,true_label,predicted_label,audio_path
0,IEMOCAP,Ses01F_impro02_M015,neu,neu,C:\Users\mateu\repos\gradient_based_speach_xai...
1,IEMOCAP,Ses01F_impro03_F000,hap,hap,C:\Users\mateu\repos\gradient_based_speach_xai...
2,IEMOCAP,Ses01F_impro01_F012,ang,ang,C:\Users\mateu\repos\gradient_based_speach_xai...
3,IEMOCAP,Ses01F_impro02_F000,sad,sad,C:\Users\mateu\repos\gradient_based_speach_xai...
4,RAVDESS,03-01-01-01-01-01-09,neu,neu,C:\Users\mateu\repos\gradient_based_speach_xai...
5,RAVDESS,03-01-03-01-01-01-03,hap,hap,C:\Users\mateu\repos\gradient_based_speach_xai...
6,RAVDESS,03-01-05-01-01-01-01,ang,ang,C:\Users\mateu\repos\gradient_based_speach_xai...
7,RAVDESS,03-01-04-01-01-01-05,sad,sad,C:\Users\mateu\repos\gradient_based_speach_xai...


Rows: 8 Correct: 8


## Run Deletion-Faithfulness Smoke Test

This smoke run uses one correctly classified audio, one deletion fraction, one random trial, and all report methods. It verifies the top/bottom/random deletion machinery and sparsity metrics for Level 3 and all LeGrad variants.

In [3]:
def run_command(command: list[str]) -> None:
    print(" ".join(str(part) for part in command))
    completed = subprocess.run(
        command,
        cwd=ROOT,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    print(completed.stdout)
    if completed.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {completed.returncode}")


deletion_root = OUTPUT_ROOT / "deletion_faithfulness"
run_command([
    sys.executable,
    "scripts/evaluate_deletion_faithfulness.py",
    "--predictions-csv", str(predictions_csv),
    "--dataset-name", "report_methods_smoke",
    "--max-examples", "1",
    "--fractions", "0.1",
    "--random-trials", "1",
    "--modes", METHODS_ARG,
    "--device", "cpu",
    "--local-files-only",
    "--output-root", str(deletion_root),
])

deletion_runs = sorted(deletion_root.glob("report_methods_smoke_deletion_faithfulness_*"), key=lambda path: path.stat().st_mtime)
deletion_run = deletion_runs[-1]
print("Deletion run:", deletion_run)


C:\Users\mateu\repos\gradient_based_speach_xai\.venv\Scripts\python.exe scripts/evaluate_deletion_faithfulness.py --predictions-csv C:\Users\mateu\repos\gradient_based_speach_xai\outputs\test_07_report_methods_smoke\report_methods_smoke_predictions.csv --dataset-name report_methods_smoke --max-examples 1 --fractions 0.1 --random-trials 1 --modes level3,legrad_final_score_relu_attention_gradient_mean_layers_source_tokens,legrad_final_score_relu_attention_gradient_renormalized_hubert_layer_weighted_source_tokens,legrad_layer_local_score_relu_attention_gradient_mean_layers_source_tokens,legrad_layer_local_score_relu_attention_gradient_renormalized_hubert_layer_weighted_source_tokens --device cpu --local-files-only --output-root C:\Users\mateu\repos\gradient_based_speach_xai\outputs\test_07_report_methods_smoke\deletion_faithfulness


Explained 1/1: 03-01-01-01-01-01-09.wav

Deletion-faithfulness evaluation complete
Examples: 1
Records: 15
Sparsity records: 5
Saved new run to: C:\Users\mateu\repos\gradient_based_speach_xai\outputs\test_07_report_methods_smoke\deletion_faithfulness\report_methods_smoke_deletion_faithfulness_20260712_004643_785966

Deletion run: C:\Users\mateu\repos\gradient_based_speach_xai\outputs\test_07_report_methods_smoke\deletion_faithfulness\report_methods_smoke_deletion_faithfulness_20260712_004643_785966


In [4]:
deletion_records = pd.read_csv(deletion_run / "deletion_records.csv")
delection_summary = pd.read_csv(deletion_run / "deletion_summary.csv")
sparsity_summary = pd.read_csv(deletion_run / "sparsity_summary_by_method.csv")
print("Deletion records:", len(deletion_records))
display(delection_summary[["explanation_mode", "strategy", "fraction", "mean_logit_drop", "mean_probability_drop"]])
display(sparsity_summary[["explanation_mode", "examples", "mean_normalized_entropy", "mean_gini", "mean_top_10_percent_mass"]])


Deletion records: 15


,explanation_mode,strategy,fraction,mean_logit_drop,mean_probability_drop
0,legrad_final_score_relu_attention_gradient_mea...,bottom,0.1,-0.008356,-0.040572
1,legrad_final_score_relu_attention_gradient_mea...,random,0.1,-0.209368,-0.066969
2,legrad_final_score_relu_attention_gradient_mea...,top,0.1,-0.067682,0.021930
3,legrad_final_score_relu_attention_gradient_ren...,bottom,0.1,-0.008356,-0.040572
4,legrad_final_score_relu_attention_gradient_ren...,random,0.1,-0.209368,-0.066969
5,legrad_final_score_relu_attention_gradient_ren...,top,0.1,-0.026638,0.032584
6,legrad_layer_local_score_relu_attention_gradie...,bottom,0.1,0.086248,0.004547
7,legrad_layer_local_score_relu_attention_gradie...,random,0.1,-0.209368,-0.066969
8,legrad_layer_local_score_relu_attention_gradie...,top,0.1,0.235368,0.055013
9,legrad_layer_local_score_relu_attention_gradie...,bottom,0.1,0.068323,-0.009283


,explanation_mode,examples,mean_normalized_entropy,mean_gini,mean_top_10_percent_mass
0,legrad_final_score_relu_attention_gradient_mea...,1,0.999347,0.046022,0.120225
1,legrad_final_score_relu_attention_gradient_ren...,1,0.999326,0.046745,0.120360
2,legrad_layer_local_score_relu_attention_gradie...,1,0.998525,0.069421,0.129223
3,legrad_layer_local_score_relu_attention_gradie...,1,0.998450,0.071213,0.129881
4,level3,1,0.859505,0.620564,0.377220


## Run Class-Specificity Smoke Test

This smoke run computes relevance for the model prediction and for the runner-up class on the same audio. Lower predicted-vs-runner-up correlation means the explanation changes more when the explained class changes.

In [5]:
class_specificity_root = OUTPUT_ROOT / "class_specificity"
run_command([
    sys.executable,
    "scripts/evaluate_class_specificity.py",
    "--predictions-csv", str(predictions_csv),
    "--dataset-name", "report_methods_smoke",
    "--max-examples", "1",
    "--modes", METHODS_ARG,
    "--device", "cpu",
    "--local-files-only",
    "--output-root", str(class_specificity_root),
])

class_specificity_runs = sorted(class_specificity_root.glob("report_methods_smoke_class_specificity_*"), key=lambda path: path.stat().st_mtime)
class_specificity_run = class_specificity_runs[-1]
print("Class-specificity run:", class_specificity_run)


C:\Users\mateu\repos\gradient_based_speach_xai\.venv\Scripts\python.exe scripts/evaluate_class_specificity.py --predictions-csv C:\Users\mateu\repos\gradient_based_speach_xai\outputs\test_07_report_methods_smoke\report_methods_smoke_predictions.csv --dataset-name report_methods_smoke --max-examples 1 --modes level3,legrad_final_score_relu_attention_gradient_mean_layers_source_tokens,legrad_final_score_relu_attention_gradient_renormalized_hubert_layer_weighted_source_tokens,legrad_layer_local_score_relu_attention_gradient_mean_layers_source_tokens,legrad_layer_local_score_relu_attention_gradient_renormalized_hubert_layer_weighted_source_tokens --device cpu --local-files-only --output-root C:\Users\mateu\repos\gradient_based_speach_xai\outputs\test_07_report_methods_smoke\class_specificity


Compared 1/1: 03-01-01-01-01-01-09.wav

Class-specificity evaluation complete
Examples: 1
Records: 5
Saved new run to: C:\Users\mateu\repos\gradient_based_speach_xai\outputs\test_07_report_methods_smoke\class_specificity\report_methods_smoke_class_specificity_20260712_004704_255100

Class-specificity run: C:\Users\mateu\repos\gradient_based_speach_xai\outputs\test_07_report_methods_smoke\class_specificity\report_methods_smoke_class_specificity_20260712_004704_255100


In [6]:
class_specificity_records = pd.read_csv(class_specificity_run / "class_specificity_records.csv")
class_specificity_summary = pd.read_csv(class_specificity_run / "class_specificity_summary_by_method.csv")
print("Class-specificity records:", len(class_specificity_records))
display(class_specificity_summary[["explanation_mode", "examples", "mean_pearson_correlation", "mean_spearman_correlation"]])


Class-specificity records: 5


,explanation_mode,examples,mean_pearson_correlation,mean_spearman_correlation
0,legrad_final_score_relu_attention_gradient_mea...,1,0.772268,0.769664
1,legrad_final_score_relu_attention_gradient_ren...,1,0.772079,0.772164
2,legrad_layer_local_score_relu_attention_gradie...,1,0.388122,0.385880
3,legrad_layer_local_score_relu_attention_gradie...,1,0.405858,0.397105
4,level3,1,-0.240824,-0.335738


## Expected Smoke-Test Counts

For `max_examples=1`, five report methods, one deletion fraction, and one random trial:

- deletion-faithfulness should produce `15` deletion rows: five methods times top/bottom/random.
- sparsity should produce `5` rows: one per method.
- class-specificity should produce `5` rows: one per method.

These are smoke-test counts only. They verify that the report attribution methods execute end-to-end; the full quantitative comparison remains the duration-matched Pastor/SpeechXAI evaluation.